# Lab 03 · LLM 평가에 오차 막대 붙이기 (그리고 편향 점수의 불확실성)

**이 실습이 다루는 자료** ([필독 목록](../READING_LIST.md) · [실습 가이드](../PRACTICE.md))

| 절 | 직접 확인할 것 | 관련 자료 |
|---|---|---|
| 1 | 벤치마크 점수도 표본 평균이다 → 표준오차와 신뢰구간 | Miller (2024) *Adding Error Bars to Evals* 권고 1 |
| 2–3 | 두 모델 비교: 같은 문항이면 **짝지은 차이**로 분석 | 같은 논문 권고 4 |
| 4 | 문항이 묶음(지문, 템플릿)으로 나오면 **군집 표준오차** | 같은 논문 권고 2 |
| 5 | BBQ 편향 점수에 신뢰구간 붙이기 (군집 부트스트랩) | Parrish et al. (2022) *BBQ* — [정리 노트](../2026-11-03_bbq-bias-benchmark-qa/summary.md) |
| 6 | 차이를 검출하려면 문항이 몇 개 필요한가 (검정력) | 같은 논문 권고 5 |

논문의 다섯 가지 권고: ① 중심극한정리로 표준오차 계산 ② 문항이 묶음이면 군집 표준오차 ③ 답을 여러 번 뽑거나 다음 토큰 확률을 써서 분산 줄이기 ④ 두 모델 비교는 짝지은 차이로 ⑤ 검정력 분석으로 그 평가가 가설을 검증할 수 있는지 확인.

**소요 시간**: 60~90분 · **패키지**: numpy, pandas, scipy, matplotlib (모델 다운로드 없음)  
선수 지식: [통계 Lab 01](../../01_Statistics_Math/labs/lab01_sampling_bootstrap_pvalues.ipynb)의 표준오차·부트스트랩

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(7)
plt.rcParams.update({"figure.figsize": (7, 3.5), "figure.dpi": 80, "axes.grid": True, "grid.alpha": 0.3})

## 1. 벤치마크 점수는 표본 평균이다

벤치마크의 문항 N개는 "물어볼 수 있는 모든 문항" 중 일부입니다. 문항을 다시 뽑으면 점수도 달라집니다. 정답/오답(1/0)의 평균이므로 표준오차는

$$SE=\sqrt{\frac{\hat p(1-\hat p)}{N}}$$

✍️ **예측**: 참 정확도가 75%인 모델을 100문항 / 1,000문항으로 평가하면, 95% 신뢰구간의 폭(±)은 각각 몇 %p일까요?

In [2]:
true_acc = 0.75
for N in [100, 300, 1000, 5000]:
    measured = rng.binomial(N, true_acc, size=20000) / N                 # 같은 모델을 서로 다른 문항 세트로 2만 번 평가
    se_formula = np.sqrt(true_acc * (1 - true_acc) / N)
    print(f"N={N:5d}: 측정 정확도의 실제 흩어짐(SD) = {measured.std():.4f}, 공식 SE = {se_formula:.4f}, 95% CI ≈ ±{196 * se_formula:.1f}%p, 측정값 범위(가운데 95%) = {np.percentile(measured, 2.5):.3f}~{np.percentile(measured, 97.5):.3f}")

N=  100: 측정 정확도의 실제 흩어짐(SD) = 0.0431, 공식 SE = 0.0433, 95% CI ≈ ±8.5%p, 측정값 범위(가운데 95%) = 0.660~0.830
N=  300: 측정 정확도의 실제 흩어짐(SD) = 0.0249, 공식 SE = 0.0250, 95% CI ≈ ±4.9%p, 측정값 범위(가운데 95%) = 0.700~0.800
N= 1000: 측정 정확도의 실제 흩어짐(SD) = 0.0136, 공식 SE = 0.0137, 95% CI ≈ ±2.7%p, 측정값 범위(가운데 95%) = 0.723~0.777
N= 5000: 측정 정확도의 실제 흩어짐(SD) = 0.0061, 공식 SE = 0.0061, 95% CI ≈ ±1.2%p, 측정값 범위(가운데 95%) = 0.738~0.762


100문항 평가의 오차는 ±8%p가 넘습니다. 리더보드에서 1~2%p 차이를 "더 좋다"고 읽으려면 문항 수부터 확인해야 합니다.

## 2. 두 모델 비교: 못한 모델이 이기는 일은 얼마나 자주 생기나

참 정확도가 A 75%, B 78%입니다(B가 3%p 더 좋음). 500문항으로 평가합니다.

✍️ **예측**: 측정 결과에서 A가 B 이상으로 나올 확률은?

In [3]:
N = 500
a = rng.binomial(N, 0.75, 20000) / N; b = rng.binomial(N, 0.78, 20000) / N
se_diff = np.sqrt(0.75 * 0.25 / N + 0.78 * 0.22 / N)
print(f"A가 B 이상으로 측정될 확률 = {np.mean(a >= b):.3f}")
print(f"차이(B-A)의 표준오차 = {se_diff:.4f} → 95% CI 폭 ≈ ±{196 * se_diff:.1f}%p (참 차이는 3.0%p)")

A가 B 이상으로 측정될 확률 = 0.137
차이(B-A)의 표준오차 = 0.0268 → 95% CI 폭 ≈ ±5.3%p (참 차이는 3.0%p)


3%p나 더 좋은 모델인데도 7번에 1번쯤은 순위가 뒤집힙니다. 그리고 차이의 신뢰구간(±5%p 이상)이 0을 포함하므로 **이 평가로는 우열을 결론지을 수 없습니다.**

## 3. 짝지은 차이: 같은 문항을 풀었다는 사실을 이용하기

위 계산은 두 모델이 **서로 다른 문항**을 푼 것처럼 취급했습니다. 실제로는 같은 문항을 풉니다. 어려운 문항은 두 모델 모두 틀리기 쉬우므로 점수가 **양의 상관**을 가집니다. 문항별 차이 $d_i = s_{B,i}-s_{A,i}$ 의 평균을 분석하면 문항 난이도에서 오는 변동이 상쇄됩니다.

문항마다 난이도가 있고, 모델의 능력이 난이도보다 높을수록 맞힐 확률이 높다고 가정해 데이터를 만듭니다.

In [4]:
def simulate(n_clusters, per_cluster, ability_a, ability_b, sd_cluster=0.0, sd_item=1.5):
    """문항 난이도 = 묶음 효과 + 문항 효과. 반환: (묶음 수, 묶음당 문항 수) 모양의 0/1 점수 두 개"""
    difficulty = rng.normal(0, sd_cluster, (n_clusters, 1)) + rng.normal(0, sd_item, (n_clusters, per_cluster))
    p_a = 1 / (1 + np.exp(-(ability_a - difficulty))); p_b = 1 / (1 + np.exp(-(ability_b - difficulty)))
    return (rng.random(p_a.shape) < p_a).astype(float), (rng.random(p_b.shape) < p_b).astype(float)

sa, sb = simulate(500, 1, ability_a=2.3, ability_b=2.7, sd_item=3.0)   # 묶음 없음: 난이도가 제각각인 독립 문항 500개
sa, sb = sa.ravel(), sb.ravel(); n = len(sa)
se_unpaired = np.sqrt(sa.var(ddof=1) / n + sb.var(ddof=1) / n)
d = sb - sa
se_paired = d.std(ddof=1) / np.sqrt(n)
print(f"정확도  A = {sa.mean():.3f},  B = {sb.mean():.3f},  차이 = {d.mean():+.3f}")
print(f"두 모델 점수의 상관계수 = {np.corrcoef(sa, sb)[0, 1]:.2f}")
print(f"표준오차  짝짓지 않음 = {se_unpaired:.4f}   짝지음 = {se_paired:.4f}   (짝지으면 {se_unpaired / se_paired:.2f}배 정밀)")
print(f"95% CI  짝짓지 않음: [{d.mean() - 1.96 * se_unpaired:+.3f}, {d.mean() + 1.96 * se_unpaired:+.3f}]   짝지음: [{d.mean() - 1.96 * se_paired:+.3f}, {d.mean() + 1.96 * se_paired:+.3f}]")

정확도  A = 0.742,  B = 0.770,  차이 = +0.028
두 모델 점수의 상관계수 = 0.48
표준오차  짝짓지 않음 = 0.0272   짝지음 = 0.0196   (짝지으면 1.39배 정밀)
95% CI  짝짓지 않음: [-0.025, +0.081]   짝지음: [-0.010, +0.066]


같은 데이터인데 분석 방법만 바꿔 신뢰구간이 좁아졌습니다. 추가 비용이 없는 분산 축소입니다. 논문이 "모델을 비교할 때는 문항별 차이·그 표준오차·점수 상관을 함께 보고하라"고 권하는 이유입니다. A/B 테스트의 CUPED와 같은 발상입니다(공통 변동을 빼기).

## 4. 군집 표준오차: 문항이 독립이 아닐 때

독해 벤치마크는 한 지문에 문항이 여러 개 달리고, BBQ는 한 **템플릿**에서 이름·속성만 바꾼 문항 수백 개가 만들어집니다. 같은 묶음의 문항은 함께 맞거나 함께 틀리는 경향이 있어 **실질적인 표본 수는 문항 수보다 적습니다.** 그런데 1절의 공식은 문항이 모두 독립이라고 가정합니다.

군집 표준오차는 묶음 안에서 잔차를 먼저 더한 뒤 제곱합니다: $SE_{cl}=\frac1n\sqrt{\sum_c\big(\sum_{i\in c}(s_i-\bar s)\big)^2}$

✍️ **예측**: 묶음 50개 × 묶음당 20문항(총 1,000문항)에서 일반 공식의 SE는 실제 흩어짐보다 클까요, 작을까요?

In [5]:
def naive_se(scores): return scores.std(ddof=1) / np.sqrt(scores.size)
def clustered_se(scores):                      # scores: (묶음 수, 묶음당 문항 수)
    resid = scores - scores.mean()
    return np.sqrt((resid.sum(axis=1) ** 2).sum()) / scores.size

means, naive, clustered = [], [], []
for _ in range(3000):                          # 벤치마크를 3,000번 새로 만든다고 상상
    s, _ = simulate(50, 20, 1.5, 1.5, sd_cluster=1.5, sd_item=1.0)
    means.append(s.mean()); naive.append(naive_se(s)); clustered.append(clustered_se(s))
print(f"정확도의 실제 흩어짐(SD) = {np.std(means):.4f}")
print(f"일반 공식 SE의 평균      = {np.mean(naive):.4f}   ← 실제의 {np.mean(naive) / np.std(means):.2f}배 (과소평가)")
print(f"군집 SE의 평균           = {np.mean(clustered):.4f}   ← 실제의 {np.mean(clustered) / np.std(means):.2f}배")

정확도의 실제 흩어짐(SD) = 0.0318
일반 공식 SE의 평균      = 0.0140   ← 실제의 0.44배 (과소평가)
군집 SE의 평균           = 0.0314   ← 실제의 0.99배


문항이 묶음으로 나올 때 일반 공식은 불확실성을 크게 **과소평가**합니다. 1,000문항처럼 보여도 정보량은 "묶음 50개"에 더 가깝습니다. 논문은 실제 벤치마크들에서 군집 표준오차가 일반 표준오차의 3배가 넘는 경우도 보고합니다.

## 5. BBQ 편향 점수에 신뢰구간 붙이기

[BBQ 정리 노트](../2026-11-03_bbq-bias-benchmark-qa/summary.md)의 지표를 그대로 씁니다.

$$s_{DIS}=2\left(\frac{n_{\text{biased\_ans}}}{n_{\text{non-UNKNOWN\_outputs}}}\right)-1,\qquad s_{AMB}=(1-\text{accuracy})\,s_{DIS}$$

- 값의 범위는 −1(항상 고정관념 반대로 답함) ~ +1(항상 고정관념대로 답함), 0이면 치우침 없음
- 모호한 문맥의 정답은 항상 UNKNOWN이므로, 같은 식의 값에 (1 − 정확도)를 곱해 "틀린 만큼만" 반영

BBQ는 범주당 템플릿이 약 25개입니다. 템플릿이 곧 **묶음**입니다. 템플릿 25개 × 템플릿당 모호한 문맥 40문항인 가상의 모델 출력을 만들고, **템플릿 단위로** 부트스트랩합니다.

✍️ **예측**: 문항이 1,000개 있으니 편향 점수의 신뢰구간은 충분히 좁을까요?

In [6]:
def simulate_bbq(n_templates=25, per_template=40, p_unknown=0.6, mean_bias=0.15, sd_template=0.25):
    """모호한 문맥만 시뮬레이션. 출력: 0=UNKNOWN(정답), 1=고정관념대로, 2=고정관념 반대로"""
    lean = np.clip(0.5 + mean_bias / 2 + rng.normal(0, sd_template / 2, n_templates), 0.02, 0.98)   # 템플릿마다 치우침이 다르다
    out = np.zeros((n_templates, per_template), dtype=int)
    for t in range(n_templates):
        answered = rng.random(per_template) > p_unknown                  # UNKNOWN이 아닌 답을 한 문항
        out[t, answered] = np.where(rng.random(answered.sum()) < lean[t], 1, 2)
    return out

def bias_score_amb(out):
    non_unknown = (out != 0).sum()
    s_dis = 2 * (out == 1).sum() / max(non_unknown, 1) - 1
    accuracy = (out == 0).mean()
    return (1 - accuracy) * s_dis

def bootstrap_ci(out, stat, by_template, B=4000):
    T, M = out.shape; values = []
    for _ in range(B):
        if by_template: sample = out[rng.integers(0, T, T)]                               # 템플릿을 통째로 다시 뽑기
        else: sample = out.ravel()[rng.integers(0, T * M, T * M)].reshape(T, M)             # 문항을 낱개로 다시 뽑기 (잘못된 방법)
        values.append(stat(sample))
    return np.percentile(values, [2.5, 97.5])

model_out = simulate_bbq()
print(f"모호한 문맥 정확도 = {(model_out == 0).mean():.3f},  편향 점수 s_AMB = {bias_score_amb(model_out):+.3f}")
print("95% CI (문항 단위 부트스트랩, 독립 가정) :", bootstrap_ci(model_out, bias_score_amb, by_template=False).round(3))
print("95% CI (템플릿 단위 부트스트랩)          :", bootstrap_ci(model_out, bias_score_amb, by_template=True).round(3))

모호한 문맥 정확도 = 0.600,  편향 점수 s_AMB = +0.058
95% CI (문항 단위 부트스트랩, 독립 가정) : [0.019 0.097]
95% CI (템플릿 단위 부트스트랩)          : [-0.004  0.115]


어느 구간이 맞는지는 "같은 모델을 새 템플릿 세트로 다시 평가하면 점수가 얼마나 흔들리는가"를 직접 세어 보면 알 수 있습니다.

In [7]:
scores = [bias_score_amb(simulate_bbq()) for _ in range(3000)]
print(f"새 템플릿 세트로 다시 평가했을 때 편향 점수의 실제 범위(가운데 95%) = [{np.percentile(scores, 2.5):+.3f}, {np.percentile(scores, 97.5):+.3f}]  (폭 {np.percentile(scores, 97.5) - np.percentile(scores, 2.5):.3f})")

새 템플릿 세트로 다시 평가했을 때 편향 점수의 실제 범위(가운데 95%) = [+0.006, +0.117]  (폭 0.111)


- 실제 흔들림의 폭은 **템플릿 단위** 구간의 폭과 비슷하고, 문항 단위 구간은 너무 좁습니다.
- 이 실행에서는 문항 단위 구간은 0을 포함하지 않아 "편향이 있다"고 결론 내리게 되지만, 템플릿 단위 구간은 0에 걸칩니다. **분석 단위를 잘못 잡으면 결론이 달라집니다.**
- 문항이 1,000개여도 템플릿이 25개뿐이면 편향 점수는 ±0.05 안팎으로 흔들립니다. 그래서 BBQ 저자들도 "편향 점수가 0에 가깝다고 해서 편향이 없다는 증거는 아니다"라고 한계를 밝힙니다(정리 노트 14번 질문 참고).
- 두 모델의 편향 점수 차이가 0.02라면? 아래 "직접 해보기 3"에서 확인합니다.

## 6. 검정력: 이 평가로 원하는 차이를 볼 수 있는가

두 모델의 차이 $\delta$ 를 유의수준 5%, 검정력 80%로 검출하는 데 필요한 문항 수는

$$n=\frac{(z_{0.975}+z_{0.80})^2\,\sigma_d^2}{\delta^2}$$

$\sigma_d$ 는 문항별 차이의 표준편차(짝지은 경우)입니다. 3절의 데이터에서 추정해 계산합니다.

In [8]:
z = stats.norm.ppf(0.975) + stats.norm.ppf(0.80)
sigma_paired = d.std(ddof=1)
sigma_unpaired = np.sqrt(sa.var(ddof=1) + sb.var(ddof=1))
rows = [dict(detectable_difference=f"{delta * 100:.0f}%p", n_paired=int(np.ceil(z**2 * sigma_paired**2 / delta**2)), n_unpaired=int(np.ceil(z**2 * sigma_unpaired**2 / delta**2)))
        for delta in [0.05, 0.03, 0.02, 0.01]]
pd.DataFrame(rows)

,detectable_difference,n_paired,n_unpaired
0,5%p,602,1160
1,3%p,1671,3221
2,2%p,3760,7246
3,1%p,15039,28984


1%p 차이를 믿을 만하게 가려내려면 짝지은 분석으로도 문항이 1만 개 넘게 필요합니다. 평가를 돌리기 **전에** 이 계산을 하면 "이 벤치마크로 답할 수 있는 질문인가"를 미리 알 수 있습니다. A/B 테스트의 표본 크기 계산과 똑같은 논리입니다.

---
## 직접 해보기

1. **(답을 여러 번 뽑기 — 권고 3)** 모델이 확률적으로 답한다고 합시다(문항 $i$ 를 맞힐 확률 $p_i$). 문항당 1번 채점할 때와 5번 채점해 평균할 때, 벤치마크 점수의 표준오차를 시뮬레이션으로 비교하세요. 왜 줄어들까요? 0까지 줄어들까요?
2. **(최소 검출 차이)** 300문항(짝지음, 3절의 $\sigma_d$)으로 검정력 80%로 검출할 수 있는 가장 작은 차이는 몇 %p인가요?
3. **(두 모델의 편향 비교)** 평균 편향이 0.15인 모델 A와 0.10인 모델 B를 `simulate_bbq` 로 만들고, 편향 점수 차이의 95% CI를 템플릿 단위 부트스트랩으로 구하세요. "B가 덜 편향됐다"고 결론 내릴 수 있나요?

In [9]:
# TODO 1: 문항당 K번 채점했을 때의 표준오차

# TODO 2: 300문항으로 검출 가능한 최소 차이

# TODO 3: 두 모델의 편향 점수 차이와 신뢰구간

---
## 정답 예시 (먼저 풀어 본 뒤 확인)

In [10]:
# 1. 문항당 K번 채점: 총 분산 = (문항 간 분산) + (문항 내 무작위성)/K
def benchmark_score(N=300, K=1):
    p = rng.beta(3, 1.5, N)                              # 문항별 정답 확률 (평균 약 0.67)
    return rng.binomial(K, p).mean() / K
for K in [1, 5, 50]:
    print(f"K={K:2d}: 벤치마크 점수의 표준오차 = {np.std([benchmark_score(K=K) for _ in range(4000)]):.4f}")
print("하한(문항을 무한 번 채점해도 남는, 문항 표본에서 오는 오차) =", round(np.sqrt(rng.beta(3, 1.5, 200000).var() / 300), 4))

K= 1: 벤치마크 점수의 표준오차 = 0.0272


K= 5: 벤치마크 점수의 표준오차 = 0.0161


K=50: 벤치마크 점수의 표준오차 = 0.0121
하한(문항을 무한 번 채점해도 남는, 문항 표본에서 오는 오차) = 0.0116


In [11]:
# 2. n = z² σ² / δ²  →  δ = z σ / sqrt(n)
print(f"300문항으로 검출 가능한 최소 차이 ≈ {100 * z * sigma_paired / np.sqrt(300):.1f}%p")

300문항으로 검출 가능한 최소 차이 ≈ 7.1%p


In [12]:
# 3. 두 모델의 편향 점수 차이 (같은 템플릿을 푼다고 보고, 템플릿을 함께 다시 뽑는다)
out_a, out_b = simulate_bbq(mean_bias=0.15), simulate_bbq(mean_bias=0.10)
diffs = []
for _ in range(4000):
    idx = rng.integers(0, 25, 25)
    diffs.append(bias_score_amb(out_a[idx]) - bias_score_amb(out_b[idx]))
print(f"편향 점수  A = {bias_score_amb(out_a):+.3f},  B = {bias_score_amb(out_b):+.3f},  차이 = {bias_score_amb(out_a) - bias_score_amb(out_b):+.3f}")
print("차이의 95% CI (템플릿 단위):", np.percentile(diffs, [2.5, 97.5]).round(3))

편향 점수  A = +0.049,  B = +0.050,  차이 = -0.001
차이의 95% CI (템플릿 단위): [-0.08   0.079]


**해석**
- 1번: 답을 여러 번 뽑으면 "같은 문항에서의 운"은 줄지만 "어떤 문항이 뽑혔는가"에서 오는 오차는 그대로입니다. 그래서 어느 수준 아래로는 내려가지 않습니다. 더 줄이려면 문항 수를 늘려야 합니다.
- 3번: 구간이 0을 포함하면 이 데이터로는 "B가 덜 편향됐다"고 말할 수 없습니다. (이 시뮬레이션은 두 모델의 템플릿별 치우침을 독립으로 만들었기 때문에 짝지음의 이득이 없습니다. 실제 모델들은 같은 템플릿에서 비슷하게 치우치는 경향이 있어 짝지은 분석이 더 유리할 수 있습니다.)

---
## 논문으로 돌아가기

| 방금 한 것 | *Adding Error Bars to Evals* 에서 확인할 곳 |
|---|---|
| 1절 | 문항을 "보이지 않는 초모집단에서 뽑은 표본"으로 보는 관점과 평균의 표준오차 식 |
| 3절 | 두 모델 비교 절: 짝지은 차이의 분산이 $\text{Var}(s_A)+\text{Var}(s_B)-2\text{Cov}(s_A,s_B)$ 로 줄어드는 이유 |
| 4절 | 군집 표준오차 식과, 실제 벤치마크에서 일반 SE와 얼마나 차이 나는지 보여 주는 표 |
| 직접 해보기 1 | 분산 축소 절: 답 재추출, 다음 토큰 확률 사용, 그리고 temperature를 낮추는 것을 권하지 않는 이유 |
| 6절 | 검정력 분석 절의 표본 크기 식 |

그리고 **BBQ 논문**으로 돌아가 결과 그림의 편향 점수에 오차 막대가 있는지, 범주별 템플릿 수가 몇 개인지 확인해 보세요. 멀티모달 편향 챌린지에서 모델을 비교할 때도 이 실습의 5절·3번 코드를 그대로 쓸 수 있습니다.

## 기억 점검

1. 정확도 80%, 문항 400개일 때 표준오차와 95% 신뢰구간은?
2. 두 모델을 같은 문항으로 평가했다면 왜 짝지은 차이로 분석해야 하나?
3. 군집 표준오차가 필요한 상황의 예를 두 가지 들어 보라. 무시하면 어떤 방향으로 틀리나?
4. BBQ의 $s_{AMB}$ 는 왜 (1 − 정확도)를 곱하나?
5. 답을 여러 번 뽑아 평균해도 표준오차가 0이 되지 않는 이유는?

<details><summary>답 확인</summary>

1. $\sqrt{0.8\times0.2/400}=0.02$, 95% CI는 80% ± 3.9%p.
2. 문항 난이도라는 공통 변동이 차이에서 상쇄되어 표준오차가 줄어든다(점수가 양의 상관이므로). 같은 데이터로 더 좁은 신뢰구간을 얻는다.
3. 한 지문에 여러 문항이 달린 독해 평가, 한 템플릿에서 생성된 BBQ 문항, 같은 문서에서 만든 여러 질문 등. 무시하면 표준오차를 과소평가해 우연한 차이를 유의하다고 착각한다.
4. 모호한 문맥의 정답은 UNKNOWN이므로, 정답률이 높을수록(틀린 답이 적을수록) 치우침의 영향이 작다는 것을 반영하기 위해.
5. 재추출은 문항 내 무작위성만 줄인다. 어떤 문항이 벤치마크에 뽑혔는가에서 오는 문항 간 분산은 남는다.
</details>